# 01 · Deterministic tools

Plain Python with unit tests: nothing here calls an LLM (plan Section 5).

| Module | What it does |
|---|---|
| `src/tools/landed_cost.py` | total = price + shipping + tax; $5 estimate when shipping is unknown |
| `src/tools/utility.py` | U = w1·P + w2·C + w3·T with the persona weights |
| `src/tools/policy.py` | budget cap, risk ceiling, reflection gate, number-check gate |
| `src/tools/prices.py` | market price for a listing's printing and variant, condition adjustment |
| `src/tools/risk.py` | risk = 1 − (1 − r_price)(1 − r_text)(1 − r_info) and the red-flag rules |

Landed cost, utility and policy are Pablo's. Changes: policy and ranking accept
`budget=None` for consultative questions (this used to crash RANKING). Prices and risk are
new; the plan lists both under `tools/`.

In [ ]:
#| default_exp tools.landed_cost

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

## Landed cost

In [ ]:
#| export
"""Deterministic landed cost calculator (Section 5)."""

from src.schemas import DeliveryType, LandedCostResult

In [ ]:
#| export
def compute_landed_cost(
    price: float,
    shipping: float | None = None,
    delivery_type: DeliveryType = "unknown",
    default_shipping_estimate: float = 5.0,
    tax_rate: float = 0.0,
) -> LandedCostResult:
    """Compute total landed cost: total = price + shipping + tax.

    Rules (Section 5):
    - Shipping is $0 only for in-person pickup.
    - When shipping is unknown (None) and not in-person, use default_shipping_estimate ($5)
      and mark shipping_estimated=True so the reply discloses the estimate.
    - Tax defaults to 0% for private local sales.
    """
    if price < 0:
        raise ValueError("price must be non-negative")
    if shipping is not None and shipping < 0:
        raise ValueError("shipping must be non-negative")
    if tax_rate < 0:
        raise ValueError("tax_rate must be non-negative")

    if delivery_type == "in_person":
        shipping_used = 0.0
        shipping_estimated = False
    elif shipping is not None:
        shipping_used = float(shipping)
        shipping_estimated = False
    else:
        shipping_used = float(default_shipping_estimate)
        shipping_estimated = True

    tax = round(price * tax_rate, 2)
    total = round(price + shipping_used + tax, 2)

    return LandedCostResult(
        price=round(float(price), 2),
        shipping=round(shipping_used, 2),
        tax=tax,
        total=total,
        shipping_estimated=shipping_estimated,
    )

## Utility

In [ ]:
#| export tools.utility
"""Deterministic persona-weighted utility scoring and ranking (Section 5)."""

from typing import Any

from src.config import load_config
from src.schemas import Condition, EvaluatedListing, Persona

In [ ]:
#| export tools.utility
def compute_utility(
    total: float,
    budget: float,
    condition: Condition,
    risk: float,
    persona: Persona,
    config: dict[str, Any] | None = None,
) -> float:
    """Compute U = w1 * P + w2 * C + w3 * T.

    - P = 1 - (total / budget), clipped to [0, 1]
    - C = condition score from config (NM 1.0, LP 0.8, MP 0.6, HP 0.4, DMG 0.2, unknown 0.5)
    - T = 1 - risk, clipped to [0, 1]
    """
    if budget <= 0:
        raise ValueError("budget must be positive")

    cfg = config or load_config()
    weights = cfg["personas"][persona]
    cond_scores = cfg["condition_utility_scores"]

    p_score = max(0.0, min(1.0, 1.0 - (total / budget)))
    c_score = float(cond_scores.get(condition, cond_scores["unknown"]))
    t_score = max(0.0, min(1.0, 1.0 - risk))

    utility = (
        weights["price"] * p_score + weights["condition"] * c_score + weights["trust"] * t_score
    )
    return round(utility, 6)

In [ ]:
#| export tools.utility
def rank_listings(
    listings: list[EvaluatedListing],
    budget: float | None,
    persona: Persona,
    config: dict[str, Any] | None = None,
) -> list[EvaluatedListing]:
    """Score and rank surviving listings. Ties go to the lower landed cost.

    With no budget (a consultative question), P is measured against the most expensive
    surviving listing, so the cheapest one gets the full price score.
    """
    cfg = config or load_config()
    scored: list[EvaluatedListing] = []
    if budget is None:
        totals = [x.landed_cost.total for x in listings if x.landed_cost and not x.rejected]
        budget = max(totals, default=0.0) or 1.0

    for item in listings:
        if item.rejected:
            continue
        if item.landed_cost is None:
            raise ValueError(
                f"Listing {item.listing.listing_id} missing landed_cost before ranking."
            )
        u = compute_utility(
            total=item.landed_cost.total,
            budget=budget,
            condition=item.listing.condition,
            risk=item.risk_score,
            persona=persona,
            config=cfg,
        )
        scored.append(item.model_copy(update={"utility_score": u}))

    # Sort descending by utility, then ascending by landed cost total (tie-breaker)
    scored.sort(
        key=lambda x: (
            -(x.utility_score if x.utility_score is not None else -1.0),
            x.landed_cost.total if x.landed_cost is not None else float("inf"),
        )
    )
    return scored

## Policy rules

In [ ]:
#| export tools.policy
"""Hard policy enforcement rules in deterministic Python (Section 5)."""

from pydantic import BaseModel

from src.schemas import EvaluatedListing

In [ ]:
#| export tools.policy
class PolicyViolationError(RuntimeError):
    """Raised when a hard execution-order policy rule is violated."""

In [ ]:
#| export tools.policy
class PolicyResult(BaseModel):
    """Structured output of policy filtering."""

    passing: list[EvaluatedListing]
    rejected: list[EvaluatedListing]
    closest_rejected: EvaluatedListing | None = None

In [ ]:
#| export tools.policy
def enforce_reflection_before_ranking(reflection_complete: bool) -> None:
    """Policy Rule 3: Refuse to rank until every candidate has been through reflection."""
    if not reflection_complete:
        raise PolicyViolationError(
            "Policy Rule 3 violated: cannot rank listings before reflection is complete."
        )

In [ ]:
#| export tools.policy
def enforce_number_check_before_speech(number_check_passed: bool) -> None:
    """Policy Rule 4: Refuse to speak until the number check passes."""
    if not number_check_passed:
        raise PolicyViolationError(
            "Policy Rule 4 violated: cannot speak response before number check passes."
        )

In [ ]:
#| export tools.policy
def evaluate_policy(
    candidates: list[EvaluatedListing],
    budget: float | None,
    risk_reject_threshold: float = 0.6,
    risk_warn_threshold: float = 0.3,
    reflection_complete: bool = True,
) -> PolicyResult:
    """Apply Policy Rules 1, 2, 3, and 5 to evaluated candidates.

    - Rule 1: Reject any listing whose landed total is over budget (skipped when budget is None,
      e.g. a consultative price question with no budget).
    - Rule 2: Reject any listing with risk >= 0.6.
    - Rule 3: Require reflection_complete=True.
    - Rule 5: If nothing passes, identify the closest rejected option, its total, and why it failed.
    """
    enforce_reflection_before_ranking(reflection_complete)

    passing: list[EvaluatedListing] = []
    rejected: list[EvaluatedListing] = []

    for item in candidates:
        if item.landed_cost is None:
            raise ValueError(
                f"Listing {item.listing.listing_id} missing landed_cost before policy check."
            )

        reasons: list[str] = []
        if item.risk_score >= risk_reject_threshold:
            reasons.append(
                f"risk score {item.risk_score:.2f} meets or exceeds ceiling {risk_reject_threshold:.2f}"
            )
        if budget is not None and item.landed_cost.total > budget:
            reasons.append(
                f"landed total ${item.landed_cost.total:.2f} exceeds budget ${budget:.2f}"
            )

        if reasons:
            rejected.append(
                item.model_copy(
                    update={
                        "rejected": True,
                        "rejection_reason": "; ".join(reasons),
                    }
                )
            )
        else:
            warning = item.warning
            if item.risk_score >= risk_warn_threshold and not warning:
                warning = f"Moderate risk ({item.risk_score:.2f}); verify photos and payment terms."
            passing.append(
                item.model_copy(
                    update={
                        "rejected": False,
                        "rejection_reason": None,
                        "warning": warning,
                    }
                )
            )

    closest_rejected: EvaluatedListing | None = None
    if not passing and rejected:
        # Prefer safe listings that only missed on budget; otherwise lowest risk then closest total.
        closest_rejected = min(
            rejected,
            key=lambda x: (
                x.risk_score >= risk_reject_threshold,
                abs((x.landed_cost.total if x.landed_cost else float("inf")) - (budget or 0.0)),
                x.risk_score,
            ),
        )

    return PolicyResult(
        passing=passing,
        rejected=rejected,
        closest_rejected=closest_rejected,
    )

## Market prices

In [ ]:
#| export tools.prices
"""Market-price lookups against the cached TCGdex prices (Section 4, step 4)."""

from typing import Any

from rapidfuzz import fuzz

from src.schemas import CardListing

In [ ]:
#| export tools.prices
def pick_variant_price(variants: dict[str, float | None], variant: str | None) -> float | None:
    """Map our variant names onto TCGplayer's variant keys (normal, holofoil, reverseHolofoil,
    1stEditionHolofoil, ...). Falls back to normal, then holofoil, then any price."""
    keys = {k.lower(): v for k, v in (variants or {}).items() if v}
    if not keys:
        return None
    if variant == "reverse":
        hits = [v for k, v in keys.items() if "reverse" in k]
    elif variant == "1st edition":
        hits = [v for k, v in keys.items() if "1st" in k or "first" in k]
    elif variant == "holo":
        hits = [v for k, v in keys.items() if "holo" in k and "reverse" not in k and "1st" not in k]
    elif variant == "normal":
        hits = [v for k, v in keys.items() if k == "normal"]
    else:
        hits = []
    if hits:
        return float(hits[0])
    for pref in ("normal", "holofoil", "holo"):
        if pref in keys:
            return float(keys[pref])
    return float(next(iter(keys.values())))

In [ ]:
#| export tools.prices
def market_price(prices: dict[str, dict], listing: CardListing) -> float | None:
    """TCGplayer market price for this listing's printing and variant, if cached."""
    entry = prices.get(listing.card_key or "")
    if not entry:
        return None
    return pick_variant_price(entry.get("variants", {}), listing.variant)

In [ ]:
#| export tools.prices
def condition_adjusted(market: float, condition: str, config: dict[str, Any]) -> float:
    """Market price scaled for condition (NM 1.0, LP 0.85, ... from config.yaml)."""
    mult = config["condition_price_multipliers"]
    return round(market * float(mult.get(condition, mult["unknown"])), 2)

In [ ]:
#| export tools.prices
def find_printing(prices: dict[str, dict], card_name: str, set_hint: str | None) -> dict | None:
    """Best price entry for a card name (and set name or id), for price-check questions."""
    best, best_score = None, 0.0
    for key, entry in prices.items():
        if key.startswith("_"):
            continue
        score = fuzz.WRatio(card_name.lower(), (entry.get("name") or "").lower())
        if set_hint:
            set_id = key.rpartition("-")[0]
            set_score = (
                100.0
                if set_hint.lower() == set_id.lower()
                else fuzz.token_set_ratio(set_hint.lower(), (entry.get("set_name") or "").lower())
            )
            score = (score + set_score) / 2
        if score > best_score:
            best, best_score = {"key": key, **entry}, score
    return best if best_score >= 80 else None

## Risk score

- **r_price**: below 40% of the condition-adjusted market price → 1.0; 40–60% → 0.5.
- **r_text**: the worst red flag matched in the title or description (high 1.0, medium 0.5),
  from `data/red_flags.md`. Word boundaries and a negation check ("not a proxy").
- **r_info**: 0.25 when there is neither a card number nor a clear photo.

Reject at ≥ 0.6, warn at ≥ 0.3 (`config.yaml`).

In [ ]:
#| export tools.risk
"""Deterministic scam-risk score (Section 5): price, red-flag text and missing-information
signals combined as risk = 1 - (1 - r_price)(1 - r_text)(1 - r_info)."""

import re
from dataclasses import dataclass, field
from typing import Any

from src.schemas import CardListing
from src.tools.prices import condition_adjusted, market_price

In [ ]:
#| export tools.risk
@dataclass
class RedFlag:
    id: str
    title: str
    severity: str  # high | medium
    patterns: list[str]
    why: str = ""
    source: str = ""

    def as_text(self) -> str:
        return f"{self.title}. Signs: {', '.join(self.patterns)}. Why it matters: {self.why}"

In [ ]:
#| export tools.risk
def parse_red_flags(markdown: str) -> list[RedFlag]:
    """Parse data/red_flags.md entries shaped like:

    ## RF-01 Payment before meeting
    - severity: high
    - patterns: zelle only, deposit first
    - why: ...
    - source: https://...
    """
    flags: list[RedFlag] = []
    for block in re.split(r"^## ", markdown, flags=re.MULTILINE)[1:]:
        lines = block.strip().splitlines()
        fid, _, title = lines[0].strip().partition(" ")
        fields: dict[str, str] = {}
        for line in lines[1:]:
            m = re.match(r"-\s*(\w+):\s*(.*)", line.strip())
            if m:
                fields[m.group(1).lower()] = m.group(2).strip()
        patterns = [p.strip().lower() for p in fields.get("patterns", "").split(",") if p.strip()]
        if patterns:
            flags.append(
                RedFlag(
                    id=fid,
                    title=title.strip(),
                    severity=fields.get("severity", "medium").lower(),
                    patterns=patterns,
                    why=fields.get("why", ""),
                    source=fields.get("source", ""),
                )
            )
    return flags

In [ ]:
#| export tools.risk
NEGATIONS = ("not a ", "not an ", "no ", "never ", "isn't a ", "is not a ")

In [ ]:
#| export tools.risk
def match_flags(text: str, red_flags: list[RedFlag]) -> list[RedFlag]:
    """Keyword rules with word boundaries and a simple negation check ("not a proxy")."""
    text = " ".join(text.lower().split())
    hits: list[RedFlag] = []
    for flag in red_flags:
        for pattern in flag.patterns:
            found = False
            for m in re.finditer(rf"(?<![a-z0-9]){re.escape(pattern)}(?![a-z0-9])", text):
                before = text[max(0, m.start() - 12) : m.start()]
                if not any(before.endswith(neg) for neg in NEGATIONS):
                    found = True
                    break
            if found:
                hits.append(flag)
                break
    return hits

In [ ]:
#| export tools.risk
@dataclass
class RiskResult:
    risk: float
    r_price: float
    r_text: float
    r_info: float
    decision: str  # reject | warn | ok
    market: float | None = None
    adjusted_market: float | None = None
    ratio: float | None = None
    matched_flags: list[str] = field(default_factory=list)  # red-flag ids
    signals: list[str] = field(default_factory=list)  # plain-language reasons

In [ ]:
#| export tools.risk
def assess(
    listing: CardListing, prices: dict, red_flags: list[RedFlag], config: dict[str, Any]
) -> RiskResult:
    th = config["thresholds"]
    signals: list[str] = []

    # Price signal, against the market price adjusted for the stated condition.
    market = market_price(prices, listing)
    r_price, adjusted, ratio = 0.0, None, None
    if market:
        adjusted = condition_adjusted(market, listing.condition, config)
        ratio = listing.price / adjusted if adjusted else None
        if ratio is not None and ratio < th["price_risk_high_ratio"]:
            r_price = 1.0
        elif ratio is not None and ratio < th["price_risk_med_ratio"]:
            r_price = 0.5
        if r_price:
            signals.append(f"price is {ratio:.0%} of the market price for its condition")
    else:
        signals.append("no market price available for this card")

    # Text signal from the red-flag knowledge base.
    flags = match_flags(f"{listing.raw_title} {listing.raw_description}", red_flags)
    severity = th["red_flag_severity"]
    r_text = max((float(severity.get(f.severity, 0.5)) for f in flags), default=0.0)
    signals += [f"red flag: {f.title}" for f in flags]

    # Missing-information signal.
    no_number = listing.number == "unknown"
    r_info = th["info_missing_penalty"] if (no_number and not listing.has_clear_photo) else 0.0
    if r_info:
        signals.append("no card number and no clear photo")

    risk = round(1 - (1 - r_price) * (1 - r_text) * (1 - r_info), 4)
    if risk >= th["risk_reject"]:
        decision = "reject"
    elif risk >= th["risk_warn"]:
        decision = "warn"
    else:
        decision = "ok"
    return RiskResult(
        risk, r_price, r_text, r_info, decision, market, adjusted, ratio,
        [f.id for f in flags], signals,
    )  # fmt: skip

# Try it

In [ ]:
from src.config import load_config
from src.pipeline import load_data
from src.tools.landed_cost import compute_landed_cost
from src.tools.risk import assess
from src.tools.utility import compute_utility

cfg = load_config()
print(compute_landed_cost(52.0, delivery_type="in_person"))
print(compute_landed_cost(52.0, shipping=None, delivery_type="shipping"))  # $5 estimate, flagged

Same listing, three personas: the Budget Parent weighs price, the Collector condition.

In [ ]:
for persona in cfg["personas"]:
    cheap_hp = compute_utility(25.0, 60.0, "HP", 0.1, persona, cfg)
    pricey_nm = compute_utility(52.0, 60.0, "NM", 0.05, persona, cfg)
    print(f"{persona:18} HP $25: {cheap_hp:.3f}   NM $52: {pricey_nm:.3f}")

Risk on the sample listings (traps are labeled scams, decoys are honest but unusual):

In [ ]:
import pandas as pd

data = load_data(cfg)
rows = []
for x in data.listings:
    r = assess(x, data.prices, data.red_flags, cfg)
    rows.append(
        {
            "id": x.listing_id,
            "label": x.label,
            "title": x.raw_title[:40],
            "price": x.price,
            "r_price": r.r_price,
            "r_text": r.r_text,
            "r_info": r.r_info,
            "risk": r.risk,
            "decision": r.decision,
        }
    )
pd.DataFrame(rows)

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()